# Integrated Intelligent Systems: Tutorial 1 - Foundations (3 Pts)

This notebook introduces the fundamental components, challenges, and definitions, central to designing and implementing Cognition-Enabled Robot Agents.

## Part 1: The Unified Theoretical Exercise (E1–E6) (1.5 Pt)

**Title:** From Natural Language Request to action execution: The Eva Coffee Scenario

![Alternative Text](coffee-task-photo.png)

### Scenario Background
Eva requests: "Bring me my coffee." Her usual blue mug is in the dishwasher; a white mug is available on the counter. The fill level can vary, and Eva may shift position during execution.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)
Students work through this unified exercise in the VRB or on paper, structured across the six developmental steps of Lecture 1:

1. **E1 · Opening Judgment (Data vs. Competence):**
   - If an end-to-end vision-language-action model or a demonstration video shows a robot successfully serving coffee 100 times in a row, does that prove the robot "can do the task" across any home environment? State your reasoning and rate your confidence from 1 to 5.
     
   ANS: No. Even if an end-to-end vision-language-action model or a demonstration video shows a robot successfully serving coffee 100 times, it does not prove that the robot can perform the task in every home environment. The demonstrations only show that the robot can succeed under the specific conditions in which it was tested. Different homes may have different layouts, furniture, obstacles, lighting, mug shapes and human positions. The robot may fail when these conditions change. True competence requires more than repeating a demonstrated action; the robot must understand the task, adapt to unfamiliar situations, predict the consequences of its actions and handle unexpected changes safely. Confidence: 5/5.

2. **E2 · Changed Case Prediction (Physical Bounds):**
   - A half-full white mug was carried successfully along the route. Suppose the mug is now nearly full. Predict the physical consequence during a turn, and identify the untested assumption in the original run.

   ANS: If the mug is nearly full, the coffee is more likely to spill during a turn because the liquid moves towards the edge due to inertia. Since there is less empty space in the mug, even a small wave can cause overflow. The untested assumption in the original run is that the mug's fill level remains half-full. The robot should not assume that the same carrying motion will be safe when the mug is nearly full. It should adjust its turning speed and trajectory to reduce spilling risk.  

3. **E3 · The Commitment Decision (Warrant Gate):**
   - The blue mug is unavailable and white-mug acceptance is unconfirmed. Choose whether the robot should ask, observe, or proceed. State the single item of evidence that justifies your choice.

   ANS: The robot should ask Eva before proceeding with the white mug. Since the blue mug is unavailable and Eva's acceptance of the white mug is not confirmed, the robot cannot assume that the substitute meets her preference. Proceeding without confirmation could result in delivering something Eva does not want. Asking her is the safest and most appropriate decision. Required evidence: Eva must explicitly confirm that she accepts the white mug as a substitute for the blue mug.   

4. **E4 · The Five Questions of Understanding (T1–T5):**
   - Formulate one distinct question for each dimension—Explain ($\text{T1}$), Predict ($\text{T2}$), Design ($\text{T3}$), Bound ($\text{T4}$), and Check ($\text{T5}$)—concerning the coffee delivery episode. At least one question must address a changed condition that a video alone cannot answer.

   ANS:  T1 – Explain: Was the coffee delivered without spilling because the robot turned slowly, or because the mug was only half-full? How did the turning speed and the amount of liquid contribute to the result?
   T2 – Predict: If the same robot carries the same mug, which is now nearly full, along the same route and makes the same turn, will the coffee spill? Why might the changed fill level affect the liquid's movement?
   T3 – Design: Would using a different gripper or placing the mug on a tray improve the safety of the delivery? How should the robot adapt its carrying speed, turning motion, or trajectory to reduce the risk of spilling?
   T4 – Bound: Under which conditions can the robot reliably deliver coffee without spilling? What combinations of mug types, liquid fill levels, carrying speeds, routes, and robot bodies are within its demonstrated capabilities, and where might it fail?
   T5 – Check: What evidence should be recorded to verify that the robot completed the delivery successfully? Does the record show Eva's acceptance of the mug, the robot's actual movements, whether any coffee was spilled, and whether Eva received the coffee?Changed condition: T2 addresses the nearly full mug. A video showing the robot successfully carrying a half-full mug cannot establish whether it will succeed with a nearly full mug. The robot must make a prediction before the changed episode and compare it with the actual observations afterwards.  

5. **E5 · System Decomposition (Mapping to Architecture):**
   - List the core functional modules required to handle Eva's request (interpreting, perceiving, proposing, predicting, warranting, acting, and recording) and map them onto the Integration Diagram.

   ANS: Functional module	Responsibility in Eva's coffee task	Corresponding part of the Integration Diagram
   1. Interpretation	Understand Eva's request and identify the coffee, preferred mug and destination. Ask Eva whether the white mug is acceptable if the blue mug is unavailable.	Goal interpretation / Task understanding
   2. Perception	Observe the environment, including the mug's location and fill level, Eva's position and possible obstacles.	Sensing and perception
   3. Proposal Generation	Generate possible actions, such as selecting the accepted mug, grasping it and choosing a route to Eva.	Planner / Action proposal
   4. Prediction	Predict the consequences of the proposed action, such as whether the coffee might spill or the robot might encounter an obstacle.	World Model / Semantic Digital Twin
   5. Warranting (Commitment Check)	Check whether there is enough evidence that the mug, grasp, route and handover satisfy the required conditions for safe execution.	Warrant Gate
   6. Acting	Execute the approved action by grasping the mug, carrying it and handing it over to Eva while monitoring the movement.	Executive / Motion Controller
   7. Recording	Store the request, assumptions, selected action, observations and final outcome for later assessment.	Episode Recorder / Audit Log

   Overall flow:

   Interpretation → Perception → Proposal Generation → Prediction → Warrant Gate → Acting → Recording.

   The Warrant Gate is important because it prevents an unverified action proposal from becoming a physical commitment.   

6. **E6 · Transfer & Exit Ticket (Justified Adaptation):**
   - Suppose Eva stands up while the robot is carrying the coffee. Revise the action, predict the potential failure mechanism, and state the exact evidence criterion required before the robot commits to the revised path.

   ANS: 1. Revised action:
   The robot should pause, keep the mug upright, and use its sensors to locate Eva's new position. It should then calculate a new safe route and handover position before continuing.
   2. Predicted potential failure:
   If the robot follows its original path, Eva or her walking frame may obstruct the route. This could cause a collision, unsafe handover, or coffee spillage due to sudden movement.
   3. Exact evidence criterion:
   Before proceeding, the robot must confirm through updated sensor observations that Eva's position is known, the revised route is obstacle-free, and the robot can reach the handover position while maintaining a stable grip and satisfying the declared safety and spill limits.
   Conclusion: The robot preserves Eva's goal but adapts its actions to the changed situation. It should only proceed when the revised action passes the commitment check.

## Part 2: Interactive Virtual Lab & Python Tutorial (1.5 Pt)

**Title:** Implementing the Embodied Cognitive Architecture

### Scenario Background
Eva requests: "Bring me my coffee."

---

In [18]:
#!/usr/bin/env python3
"""Embodied Cognitive Architecture
"""

from dataclasses import dataclass, field
from typing import Dict, Any, Tuple, Optional, List
import random
import time

@dataclass(frozen=True)
class GoalSpecification:
    """Formalized, concrete goal specification derived via semantic interpretation."""
    target_object: str
    destination: str
    target_volume_ml: float
    max_allowable_force_n: float
    max_slosh_wave_mm: float
    max_nav_velocity_mps: float
    time_frame_seconds: float

@dataclass
class ParameterPacket:
    """System 1 (The Proposer): Flexible sampler proposing execution parameters."""
    grasp_pose: Tuple[float, float, float]
    max_velocity_mps: float
    tilt_rate_radps: float
    contact_depth_m: float = 0.01
    source_generator: str = "stochastic_sampler"
    estimated_battery_drain_percent: float = 16.0


class Proposer:
    """Generates candidate action parameters using uniform sampling."""

    def __init__(self):
        self.min_velocity = 0.1
        self.max_velocity = 0.6
        self.min_tilt_rate = 0.01
        self.max_tilt_rate = 0.15

    def propose(self) -> ParameterPacket:
        velocity = random.uniform(
            self.min_velocity,
            self.max_velocity
        )

        tilt_rate = random.uniform(
            self.min_tilt_rate,
            self.max_tilt_rate
        )

        grasp_pose = (
            random.uniform(0.3, 0.5),
            random.uniform(0.1, 0.2),
            random.uniform(0.2, 0.4)
        )

        return ParameterPacket(
            grasp_pose=grasp_pose,
            max_velocity_mps=velocity,
            tilt_rate_radps=tilt_rate,
            source_generator="uniform_sampler"
        )


@dataclass
class Observation:
    """Raw, noisy sensor measurements at time t (distinct from internal belief)."""
    timestamp: float
    measured_temperature_c: float
    measured_friction: float
    eva_position: str
    room_busyness: str

@dataclass
class BeliefState:
    """The robot's current internal belief state (updated via perception)."""
    pouch_present: bool = True
    pouch_material: str = "flexible_plastic"
    fluid_viscosity_pas: float = 1.2
    vial_ready: bool = True
    camera_active: bool = True
    fallback_path_exists: bool = True
    audit_logger_enabled: bool = True
    # Real-time environmental parameters maintained internally
    ambient_temperature_c: float = 21.5
    surface_friction_coefficient: float = 0.45
    eva_position: str = "living_room"
    room_busyness: str = "medium"  # 'low', 'medium', 'high'
    preferred_cup: str = "blue_ceramic_mug"
    battery_level_percent: float = 15.0


class InterpretationSystem:
    """Uses a semantic knowledge base to map vague instructions and context 
    into precise physical goal parameters (GoalSpecification).
    """
    def __init__(self, knowledge_base: Dict[str, Any]):
        self.kb = knowledge_base

    def interpret(self, raw_utterance: str, belief: BeliefState) -> GoalSpecification:
        print(f"\n[INTERPRETATION] Analyzing instruction: '{raw_utterance}' in context of Eva (Position: {belief.eva_position}, Room Busyness: {belief.room_busyness})")
        
        cup_type = belief.preferred_cup
        cup_properties = self.kb["cups"].get(cup_type, {"max_force": 5.0, "max_wave": 2.0})
        
        busyness_factor = self.kb["room_busyness_limits"].get(belief.room_busyness, 0.5)
        max_nav_vel = 1.2 * busyness_factor
        
        is_urgent = "schnell" in raw_utterance.lower() or "fast" in raw_utterance.lower()
        time_frame = 30.0 if is_urgent else 60.0
        target_volume = 15.0 if not is_urgent else 12.0
        
        goal = GoalSpecification(
            target_object=cup_type,
            destination=belief.eva_position,
            target_volume_ml=target_volume,
            max_allowable_force_n=cup_properties["max_force"],
            max_slosh_wave_mm=cup_properties["max_wave"],
            max_nav_velocity_mps=max_nav_vel,
            time_frame_seconds=time_frame
        )
        
        print(f" -> Derived Goal: Object={goal.target_object}, MaxForce={goal.max_allowable_force_n}N, MaxNavVel={goal.max_nav_velocity_mps}m/s, TimeFrame={goal.time_frame_seconds}s")
        return goal


class SensingSystem:
    """Simulates real-time noisy hardware sensors emitting raw observations."""
    def __init__(self):
        pass

    def sense(self, true_ambient_temp: float, true_friction: float, eva_pos: str, busyness: str) -> Observation:
        noisy_temp = true_ambient_temp + random.uniform(-0.5, 0.5)
        noisy_friction = true_friction + random.uniform(-0.05, 0.05)
        
        return Observation(
            timestamp=time.time(),
            measured_temperature_c=round(noisy_temp, 2),
            measured_friction=round(noisy_friction, 2),
            eva_position=eva_pos,
            room_busyness=busyness
        )


class PerceptionSystem:
    """Estimator module that updates the internal BeliefState using raw Observations."""
    def __init__(self, belief_state: BeliefState):
        self.belief_state = belief_state

    def update(self, observation: Observation) -> None:
        self.belief_state.ambient_temperature_c = observation.measured_temperature_c
        self.belief_state.surface_friction_coefficient = observation.measured_friction
        self.belief_state.eva_position = observation.eva_position
        self.belief_state.room_busyness = observation.room_busyness


class SemanticDigitalTwin:
    """System 2 (Semantic Digital Twin): Imperfect models with sim-to-real epistemic gaps."""
    def __init__(self, belief_state: BeliefState):
        self.belief = belief_state

    

    def predict_properties(
        self, params: ParameterPacket, goal: GoalSpecification
    ) -> Dict[str, float]:

        # Stage 1: Normal contact stiffness
        stiffness_coefficient = (
            12.5 * (1.0 + self.belief.surface_friction_coefficient * 0.1)
        )

        # Stage 2: Increased stiffness after bottoming-out
        if params.contact_depth_m > 0.02:
            stiffness_coefficient *= 1.8

        # Calculate predicted contact force
        predicted_force = (
            stiffness_coefficient * (params.max_velocity_mps ** 1.5)
        )

        # Temperature-dependent fluid viscosity
        reference_temperature_c = 20.0
        thermal_degradation_coefficient = 0.01

        temperature_difference = (
            self.belief.ambient_temperature_c - reference_temperature_c
        )

        effective_viscosity = self.belief.fluid_viscosity_pas * (
            1.0 - thermal_degradation_coefficient * temperature_difference
        )

        effective_viscosity = max(effective_viscosity, 0.01)

        radius = 0.05
        wave_height_m = (
            params.tilt_rate_radps * radius
        ) / effective_viscosity

        predicted_wave = wave_height_m * 1000.0

        # Sim-to-real epistemic prediction error
        epistemic_force_error = random.uniform(-0.2, 0.3)
        epistemic_wave_error = random.uniform(-0.1, 0.2)

        return {
            "predicted_force_n": round(
                max(0.0, predicted_force + epistemic_force_error), 2
            ),
            "predicted_wave_mm": round(
                max(0.0, predicted_wave + epistemic_wave_error), 2
            ),
            "predicted_nav_velocity": min(
                params.max_velocity_mps, goal.max_nav_velocity_mps
            ),
        }



class WarrantGate:
    """Validates predicted properties against the interpreted GoalSpecification."""
    def __init__(self, sdt: SemanticDigitalTwin):
        self.sdt = sdt

    def validate(self, params: ParameterPacket, goal: GoalSpecification, belief: BeliefState) -> Dict[str, Any]:
        predictions = self.sdt.predict_properties(params, goal)
        
        checks = {
            "grounded": belief.vial_ready and belief.pouch_present,
            "feasible": params.max_velocity_mps > 0.0 and params.tilt_rate_radps > 0.0,
            "safe": (predictions["predicted_force_n"] <= goal.max_allowable_force_n) and 
                    (predictions["predicted_wave_mm"] <= goal.max_slosh_wave_mm) and
                    (predictions["predicted_nav_velocity"] <= goal.max_nav_velocity_mps),
            "energetically_viable": (params.estimated_battery_drain_percent < 15.0 
                                     and params.estimated_battery_drain_percent
                                     <= belief.battery_level_percent
                                    ),
            "monitorable": belief.camera_active,
            "repairable": belief.fallback_path_exists,
            "recordable": belief.audit_logger_enabled
        }
        
        all_passed = all(checks.values())
        failed_criterion = next((name for name, passed in checks.items() if not passed), None)
        
        return {
            "passed": all_passed,
            "failed_criterion": failed_criterion,
            "predictions": predictions,
            "warrant_checks": checks
        }


class ExecutionEngine:
    """Manages physical execution and constructs the epistemic audit trail."""
    def __init__(self):
        self.audit_trail: List[Dict[str, Any]] = []

    def execute_action(self, params: ParameterPacket, goal: GoalSpecification, warrant_result: Dict[str, Any]) -> Dict[str, Any]:
        print(f"\n[ACTUATOR] Executing motion in physical world: v_max={params.max_velocity_mps} m/s")
        time.sleep(0.1)
        
        # Real-world divergence check
        predictions = warrant_result["predictions"]
        actual_force = predictions["predicted_force_n"] + random.uniform(-0.3, 0.6)
        actual_wave = predictions["predicted_wave_mm"] + random.uniform(-0.2, 0.5)
        
        spill_occurred = (actual_force > goal.max_allowable_force_n) or (actual_wave > goal.max_slosh_wave_mm)
        status = "FAILURE" if spill_occurred else "SUCCESS"
        
        execution_outcome = {
            "actual_force_n": round(actual_force, 2),
            "actual_wave_mm": round(actual_wave, 2),
            "actual_volume_transferred_ml": goal.target_volume_ml if not spill_occurred else goal.target_volume_ml - 3.0,
            "spill_occurred": spill_occurred,
            "status": status
        }
        return execution_outcome
    def record_trace(
        self, raw_command, goal, belief, proposal,
        warrant, observation, committed=False
):
        trace = {
            "asked": {
                "raw_utterance": raw_command,
                "interpreter_output": {
                    "target": goal.target_object,
                    "max_force_limit_n": goal.max_allowable_force_n,
                    "max_nav_velocity": goal.max_nav_velocity_mps
            }
        },

                "belief": {
                    "eva_position": belief.eva_position,
                    "room_busyness": belief.room_busyness,
                    "ambient_temp_c": belief.ambient_temperature_c,
                    "surface_friction": belief.surface_friction_coefficient,
                    "battery_level_percent": belief.battery_level_percent
                },

                "committed": {
                    "action_approved": committed,
                    "parameters": {
                        "max_velocity_mps": proposal.max_velocity_mps,
                        "tilt_rate_radps": proposal.tilt_rate_radps,
                        "contact_depth_m": proposal.contact_depth_m
                    },
                    "warrant_verdict": warrant
                },

                "observed": {
                    "timestamp": observation.timestamp,
                    "measured_temp_c": observation.measured_temperature_c,
                    "measured_friction": observation.measured_friction,
                    "eva_position": observation.eva_position,
                    "room_busyness": observation.room_busyness
                },

                "execution_outcome": None
            }

        self.audit_trail.append(trace)
        
def audit_analyzer(audit_trail):
    approved = 0
    rejected = 0
    failed_criteria = []
    force_errors = []

    for trace in audit_trail:
        committed = trace["committed"]
        warrant = committed["warrant_verdict"]

        if committed["action_approved"]:
            approved += 1
        else:
            rejected += 1

        for criterion, passed in warrant["warrant_checks"].items():
            if not passed:
                failed_criteria.append(criterion)

        outcome = trace.get("execution_outcome")

        if outcome is not None:
            predicted_force = warrant["predictions"]["predicted_force_n"]
            actual_force = outcome["actual_force_n"]
            force_errors.append(abs(actual_force - predicted_force))

    counts = {
        criterion: failed_criteria.count(criterion)
        for criterion in set(failed_criteria)
    }

    if counts:
        highest_count = max(counts.values())
        most_frequent_failure = [
            criterion
            for criterion, count in counts.items()
            if count == highest_count
        ]
    else:
        most_frequent_failure = []

    import statistics

    average_force_variance = (
        round(statistics.pvariance(force_errors), 3)
        if force_errors else None
    )

    return {
        "approved_proposals": approved,
        "rejected_proposals": rejected,
        "most_frequent_failed_criterion": most_frequent_failure,
        "failed_criterion_frequency": counts,
        "average_force_prediction_error_variance": average_force_variance
    }
# --- Execution Demonstration ---
if __name__ == "__main__":
    knowledge_base = {
        "cups": {
            "blue_ceramic_mug": {"max_force": 4.5, "max_wave": 1.5},
            "plastic_cup": {"max_force": 7.0, "max_wave": 3.0}
        },
        "room_busyness_limits": {
            "low": 1.0,
            "medium": 0.6,
            "high": 0.3
        }
    }

    # 1. Initialize systems
    belief_state = BeliefState(room_busyness="medium", preferred_cup="blue_ceramic_mug")
    sensing = SensingSystem()
    perception = PerceptionSystem(belief_state)
    interpreter = InterpretationSystem(knowledge_base)
    sdt = SemanticDigitalTwin(belief_state)
    warrant_gate = WarrantGate(sdt)
    executive = ExecutionEngine()
    
    print("=== STEP 1: Sensing & Perception Update ===")
    raw_observation = sensing.sense(
        true_ambient_temp=21.5, 
        true_friction=0.45, 
        eva_pos="living_room", 
        busyness="medium"
    )
    perception.update(raw_observation)
    print(f"Raw Sensor Observation : {raw_observation}")
    print(f"Updated Internal Belief : Temp={belief_state.ambient_temperature_c}°C, Friction={belief_state.surface_friction_coefficient}")
    
    print("\n=== STEP 2: Instruction Interpretation ===")
    raw_command = "Bring me my coffee fast!"
    goal = interpreter.interpret(raw_command, belief_state)
    
    print("\n=== STEP 3: System 1 Proposer Generates Parameters ===")
    proposal_v1 = ParameterPacket(
        grasp_pose=(0.45, 0.12, 0.30),
        max_velocity_mps=0.8, 
        tilt_rate_radps=0.2
    )
    print(f"Proposer suggests: v_max = {proposal_v1.max_velocity_mps} m/s")
    
    print("\n=== STEP 4: System 2 SDT & Warrant Gate Validation (Attempt 1) ===")
    warrant_v1 = warrant_gate.validate(proposal_v1, goal, belief_state)
    print(f"SDT Predictions: Nav-Vel = {warrant_v1['predictions']['predicted_nav_velocity']} m/s")
    print(f"Warrant Checks  : {warrant_v1['warrant_checks']}")
    print(f"Gate Verdict    : REJECTED (Failed on: {warrant_v1['failed_criterion']})")
    executive.record_trace(
    raw_command, goal, belief_state,
    proposal_v1, warrant_v1, raw_observation,
    committed=False
    )
    
    
    print("\n=== STEP 5: Proposer Resamples (Attempt 2) ===")
    proposal_v2 = ParameterPacket(
        grasp_pose=(0.45, 0.12, 0.30),
        max_velocity_mps=0.45, 
        tilt_rate_radps=0.08
    )
    
    warrant_v2 = warrant_gate.validate(proposal_v2, goal, belief_state)
    print(f"SDT Predictions: Force = {warrant_v2['predictions']['predicted_force_n']} N, Wave = {warrant_v2['predictions']['predicted_wave_mm']} mm")
    print(f"Warrant Checks  : {warrant_v2['warrant_checks']}")
    if warrant_v2["passed"]:
        print("Gate Verdict    : APPROVED")
    else:
        print(
            f"Gate Verdict    : REJECTED "
            f"(Failed on: {warrant_v2['failed_criterion']})"
        )
    executive.record_trace(
        raw_command, goal, belief_state,
        proposal_v2, warrant_v2, raw_observation,
        committed=False
    )
    print("\n=== STEP 6: Physical Execution & Epistemic Audit Trail ===")
    if warrant_v2["passed"]:
        final_outcome = executive.execute_action(
            proposal_v2, goal, warrant_v2
    )

        executive.audit_trail[-1]["committed"]["action_approved"] = True
        executive.audit_trail[-1]["execution_outcome"] = final_outcome

        print(f"Execution Outcome: {final_outcome}")

    print(f"\n--- COMPLETE EPISODIC AUDIT TRAIL ({len(executive.audit_trail)} Entries) ---")
    for idx, trace in enumerate(executive.audit_trail, 1):
        status = trace['committed']['action_approved']
        outcome = trace.get('execution_outcome')
        outcome_str = f" | Outcome: {outcome['status']} (Spill: {outcome['spill_occurred']})" if outcome else ""
        print(f"Entry #{idx} | Approved: {status}{outcome_str} | Command: '{trace['asked']['raw_utterance']}'")

=== STEP 1: Sensing & Perception Update ===
Raw Sensor Observation : Observation(timestamp=1793611096.9571428, measured_temperature_c=21.25, measured_friction=0.49, eva_position='living_room', room_busyness='medium')
Updated Internal Belief : Temp=21.25°C, Friction=0.49

=== STEP 2: Instruction Interpretation ===

[INTERPRETATION] Analyzing instruction: 'Bring me my coffee fast!' in context of Eva (Position: living_room, Room Busyness: medium)
 -> Derived Goal: Object=blue_ceramic_mug, MaxForce=4.5N, MaxNavVel=0.72m/s, TimeFrame=30.0s

=== STEP 3: System 1 Proposer Generates Parameters ===
Proposer suggests: v_max = 0.8 m/s

=== STEP 4: System 2 SDT & Warrant Gate Validation (Attempt 1) ===
SDT Predictions: Nav-Vel = 0.72 m/s
Warrant Checks  : {'grounded': True, 'feasible': True, 'safe': False, 'energetically_viable': False, 'monitorable': True, 'repairable': True, 'recordable': True}
Gate Verdict    : REJECTED (Failed on: safe)

=== STEP 5: Proposer Resamples (Attempt 2) ===
SDT Predi

In [2]:
print("Audit trail length:", len(executive.audit_trail))
audit_analyzer(executive.audit_trail)

print("record_trace:", hasattr(executive, "record_trace"))
print("audit_analyzer:", callable(audit_analyzer))

Audit trail length: 2
record_trace: True
audit_analyzer: True


In [9]:

proposer = Proposer()

for i in range(5):
    proposal = proposer.propose()
    print(f"Proposal {i+1}: {proposal}")


Proposal 1: ParameterPacket(grasp_pose=(0.48755999527529464, 0.17091670510259724, 0.2315364753038941), max_velocity_mps=0.2895530742206659, tilt_rate_radps=0.07315489880465702, contact_depth_m=0.01, source_generator='uniform_sampler', estimated_battery_drain_percent=16.0)
Proposal 2: ParameterPacket(grasp_pose=(0.47374497406753374, 0.1042620162370187, 0.2621294095191945), max_velocity_mps=0.19847871023618888, tilt_rate_radps=0.031645741008548035, contact_depth_m=0.01, source_generator='uniform_sampler', estimated_battery_drain_percent=16.0)
Proposal 3: ParameterPacket(grasp_pose=(0.4335642757841522, 0.19107621937683675, 0.21537240423463203), max_velocity_mps=0.16848132945176278, tilt_rate_radps=0.13988320675146204, contact_depth_m=0.01, source_generator='uniform_sampler', estimated_battery_drain_percent=16.0)
Proposal 4: ParameterPacket(grasp_pose=(0.4500417948849189, 0.10556544176508637, 0.2823686950566486), max_velocity_mps=0.36191796522555075, tilt_rate_radps=0.11179938356012531, co

In [10]:

# Task 2.2: Aggressive contact test

aggressive_proposal = ParameterPacket(
    grasp_pose=(0.45, 0.12, 0.30),
    max_velocity_mps=0.35,
    tilt_rate_radps=0.1,
    contact_depth_m=0.025
)

# Validate through the Warrant Gate
aggressive_result = warrant_gate.validate(
    aggressive_proposal, goal, belief_state
)

print("=== TASK 2.2: AGGRESSIVE CONTACT TEST ===")
print("Predicted force:",
      aggressive_result["predictions"]["predicted_force_n"], "N")

print("Warrant checks:",
      aggressive_result["warrant_checks"])

print("Gate passed:", aggressive_result["passed"])
print("Failed criterion:",
      aggressive_result["failed_criterion"])

if not aggressive_result["passed"]:
    print("REJECTED: Physical execution must not proceed.")
else:
    print("APPROVED: Proposal may proceed to execution.")


=== TASK 2.2: AGGRESSIVE CONTACT TEST ===
Predicted force: 4.77 N
Warrant checks: {'grounded': True, 'feasible': True, 'safe': False, 'energetically_viable': False, 'monitorable': True, 'repairable': True, 'recordable': True}
Gate passed: False
Failed criterion: safe
REJECTED: Physical execution must not proceed.


In [11]:
# Q3.2: Compare wave height at different temperatures

test_tilt_rate = 0.15

# Keep the same tilt rate and compare two temperatures
original_temperature = belief_state.ambient_temperature_c

params = ParameterPacket(
    grasp_pose=(0.45, 0.12, 0.30),
    max_velocity_mps=0.45,
    tilt_rate_radps=test_tilt_rate,
    contact_depth_m=0.01
)

# Prediction at reference temperature
belief_state.ambient_temperature_c = 20.0
reference_result = sdt.predict_properties(params, goal)

# Prediction at higher temperature
belief_state.ambient_temperature_c = 28.0
hot_result = sdt.predict_properties(params, goal)

# Restore original temperature
belief_state.ambient_temperature_c = original_temperature

print("=== TASK 3.2: TEMPERATURE AND FLUID SLOSHING ===")
print("Tilt rate:", test_tilt_rate, "rad/s")
print("Wave at 20°C:",
      reference_result["predicted_wave_mm"], "mm")
print("Wave at 28°C:",
      hot_result["predicted_wave_mm"], "mm")

=== TASK 3.2: TEMPERATURE AND FLUID SLOSHING ===
Tilt rate: 0.15 rad/s
Wave at 20°C: 6.26 mm
Wave at 28°C: 6.94 mm


In [12]:
# TASK 4.2: LOW-BATTERY TEST

belief_state.battery_level_percent = 15.0

test_proposal = ParameterPacket(
    grasp_pose=(0.45, 0.12, 0.30),
    max_velocity_mps=0.45,
    tilt_rate_radps=0.08,
    contact_depth_m=0.01,
    estimated_battery_drain_percent=16.0
)

energy_result = warrant_gate.validate(
    test_proposal, goal, belief_state
)

print("=== TASK 4.2: LOW-BATTERY TEST ===")
print("Battery level:", belief_state.battery_level_percent, "%")
print("Warrant checks:", energy_result["warrant_checks"])
print("Gate passed:", energy_result["passed"])
failed_criteria = [
    name
    for name, passed in energy_result["warrant_checks"].items()
    if not passed
]

print("All failed criteria:", failed_criteria)

if not energy_result["warrant_checks"]["energetically_viable"]:
    print("REJECTED: Redirecting robot to recharge.")
    print("Coffee delivery postponed until sufficient battery is available.")

=== TASK 4.2: LOW-BATTERY TEST ===
Battery level: 15.0 %
Warrant checks: {'grounded': True, 'feasible': True, 'safe': False, 'energetically_viable': False, 'monitorable': True, 'repairable': True, 'recordable': True}
Gate passed: False
All failed criteria: ['safe', 'energetically_viable']
REJECTED: Redirecting robot to recharge.
Coffee delivery postponed until sufficient battery is available.


In [ ]:
# Verify the ExecutionEngine created in the main execution cell
print("Audit trail length:", len(executive.audit_trail))
print("record_trace:", hasattr(executive, "record_trace"))
print("audit_analyzer:", callable(audit_analyzer))

record_trace: True
audit_analyzer: True


In [14]:
# Verify the existing ExecutionEngine without resetting its audit trail
print("Audit trail length:", len(executive.audit_trail))
print("record_trace:", hasattr(executive, "record_trace"))
print("audit_analyzer:", callable(audit_analyzer))

Audit trail length: 0
record_trace: True
audit_analyzer: True


In [15]:
print("Current audit trail length:", len(executive.audit_trail))

for i, trace in enumerate(executive.audit_trail, 1):
    print(
        i,
        trace["asked"]["raw_utterance"],
        trace["committed"]["action_approved"]
    )

Current audit trail length: 2
1 Bring me my coffee fast! False
2 Bring me my coffee fast! False


In [15]:

print("Audit trail length:", len(executive.audit_trail))
audit_analyzer(executive.audit_trail)

Audit trail length: 0


{'approved_proposals': 0,
 'rejected_proposals': 0,
 'most_frequent_failed_criterion': 'None',
 'failed_criterion_frequency': {},
 'average_absolute_force_prediction_error_n': None}

In [17]:
analysis = audit_analyzer(executive.audit_trail)

print("Raw analyzer result:", analysis)
print("Audit trail length:", len(executive.audit_trail))

Raw analyzer result: {'approved_proposals': 0, 'rejected_proposals': 1, 'most_frequent_failed_criterion': ['energetically_viable', 'safe'], 'failed_criterion_frequency': {'energetically_viable': 1, 'safe': 1}, 'average_force_prediction_error_variance': None}
Audit trail length: 2


In [19]:
print("=== TASK 5.1: AUDIT ANALYSIS ===")

analysis = audit_analyzer(executive.audit_trail)

print("Approved:", analysis["approved_proposals"])
print("Rejected:", analysis["rejected_proposals"])
print("Most frequent failed criteria:",
      analysis["most_frequent_failed_criterion"])
print("Failure frequency:", analysis["failed_criterion_frequency"])
print("Average force prediction error variance:",
      analysis["average_force_prediction_error_variance"])

=== TASK 5.1: AUDIT ANALYSIS ===
Approved: 0
Rejected: 2
Most frequent failed criteria: ['energetically_viable', 'safe']
Failure frequency: {'energetically_viable': 2, 'safe': 2}
Average force prediction error variance: None


In [20]:
print("=== TASK 5.2: EPISTEMIC AUDIT TRACE ===")

for i, trace in enumerate(executive.audit_trail, 1):
    print(f"\nEntry {i}")
    print("Asked:", trace.get("asked"))
    print("Belief:", trace.get("belief"))
    print("Committed:", trace.get("committed"))
    print("Observed:", trace.get("observed"))

=== TASK 5.2: EPISTEMIC AUDIT TRACE ===

Entry 1
Asked: {'raw_utterance': 'Bring me my coffee fast!', 'interpreter_output': {'target': 'blue_ceramic_mug', 'max_force_limit_n': 4.5, 'max_nav_velocity': 0.72}}
Belief: {'eva_position': 'living_room', 'room_busyness': 'medium', 'ambient_temp_c': 21.25, 'surface_friction': 0.49, 'battery_level_percent': 15.0}
Committed: {'action_approved': False, 'parameters': {'max_velocity_mps': 0.8, 'tilt_rate_radps': 0.2, 'contact_depth_m': 0.01}, 'warrant_verdict': {'passed': False, 'failed_criterion': 'safe', 'predictions': {'predicted_force_n': 9.65, 'predicted_wave_mm': 8.6, 'predicted_nav_velocity': 0.72}, 'warrant_checks': {'grounded': True, 'feasible': True, 'safe': False, 'energetically_viable': False, 'monitorable': True, 'repairable': True, 'recordable': True}}}
Observed: {'timestamp': 1793611096.9571428, 'measured_temp_c': 21.25, 'measured_friction': 0.49, 'eva_position': 'living_room', 'room_busyness': 'medium'}

Entry 2
Asked: {'raw_utteran

## Questions (0.25 Pt x 4 + 0.5 Pt)

### Task 1: Architectural Mapping & Code Decomposition
* **Question 1.1:** Map the classes and methods implemented in your Python script directly onto the functional blocks of the lecture integration diagram. Explain how data flows between these components during a single cognitive decision cycle.
* **Question 1.2:** Complete the proposer class based on a probabilistic model of your choice (e.g., uniform, gaussian, mixture). Justify your choice.

ANS: The Python implementation follows a modular cognitive architecture. Each class performs a specific function, and the modules exchange information to complete a cognitive decision cycle.
Functional block	Python class / method	Responsibility
Interpretation	InterpretationSystem.interpret()	Converts Eva's natural-language request into a formal GoalSpecification.
Sensing	SensingSystem.sense()	Collects noisy environmental measurements such as temperature and friction.
Perception	PerceptionSystem.update()	Processes observations and updates the internal BeliefState.
Proposal Generation	Proposer.propose()	Generates candidate movement parameters.
World Model / Semantic Digital Twin	SemanticDigitalTwin.predict_properties()	Predicts contact force, fluid wave height and navigation velocity.
Warrant Gate	WarrantGate.validate()	Checks whether the proposed action satisfies safety and execution requirements.
Executive / Motion Controller	ExecutionEngine.execute_action()	Executes an approved physical action.
Episode Recorder / Audit Log	ExecutionEngine.record_trace()	Records the request, belief, observations, decision and execution outcome.

Data flow during one cognitive decision cycle:

1. Eva gives the instruction: "Bring me my coffee fast!"
2. The Sensing System collects environmental information, such as temperature, friction and Eva's position.
3. The Perception System processes these observations and updates the robot's internal BeliefState.
4. The Interpretation System uses the instruction and updated belief to generate a structured GoalSpecification.
5. The Proposer generates candidate action parameters.
6. The Semantic Digital Twin predicts the physical consequences of the proposed action.
7. The Warrant Gate evaluates the predictions against the goal and system constraints.
8. If approved, the Executive executes the action. Otherwise, the proposal is rejected.
9. The Episode Recorder stores the request, belief, observations, decision and execution outcome for later auditing.

The Warrant Gate acts as a commitment boundary: a proposed action cannot proceed to physical execution unless it passes the required checks.

### Task 2: Non-Linear Contact Mechanics and Material Compliance
In the base script, the contact force on the flexible pouch is modeled using a simplified power-law equation. 

* **Question 2.1:** Modify the `SemanticDigitalTwin.predict_properties` method to account for a dual-stage material compliance curve. If the contact depth exceeds $0.02 \, \text{m}$, the stiffness coefficient should increase exponentially by a factor of $1.8$ due to bottoming out against a rigid internal fixture.

ANS: I modified the `SemanticDigitalTwin.predict_properties()` method to implement a dual-stage contact compliance model. In the first stage, the stiffness coefficient is calculated based on the surface friction coefficient, representing normal contact conditions. In the second stage, when the contact depth exceeds 0.02 m, the stiffness coefficient is multiplied by 1.8 to represent the increased resistance caused by bottoming out against a rigid internal fixture. The predicted contact force is then calculated using:

Predicted Force = Stiffness Coefficient × (Approach Velocity ^ 1.5) 

This modification allows the Semantic Digital Twin to account for increased contact resistance at greater contact depths and predict potentially unsafe contact forces before physical execution.

* **Question 2.2:** Write a test case where the Proposer generates an aggressive approach velocity ($0.35 \, \text{m/s}$). Calculate the predicted force and verify that the Warrant Gate successfully intercepts and rejects the parameter packet before physical execution.

ANS: I tested an aggressive contact proposal using an approach velocity of 0.35 m/s and a contact depth of 0.025 m. The Semantic Digital Twin predicted a contact force of 4.77 N, exceeding the blue ceramic mug's maximum allowable force of 4.5 N. Consequently, the safety criterion failed and the Warrant Gate rejected the proposal. The energy viability criterion also failed, preventing physical execution. This demonstrates how the Warrant Gate evaluates proposed actions before allowing physical commitment.

### Task 3: Advanced Fluid Sloshing and Viscosity Temperature Coupling
Fluids in laboratory automation often exhibit temperature-dependent viscosity changes.
* **Question 3.1:** Enhance the fluid dynamics model so that ambient temperature fluctuations directly alter the effective fluid viscosity using an Arrhenius-type approximation or a linear thermal degradation coefficient ($1\% \, \text{drop per} \, ^\circ\text{C}$ above $20^\circ\text{C}$).

ANS: I modified the fluid dynamics model in `SemanticDigitalTwin.predict_properties()` to incorporate temperature-dependent viscosity using a linear thermal degradation approximation. The model uses 20°C as the reference temperature and a thermal degradation coefficient of 0.01. As the ambient temperature increases above the reference temperature, the effective fluid viscosity decreases. The calculated effective viscosity is then used in the wave-height prediction. A minimum viscosity value of 0.01 Pa·s is maintained to prevent zero or negative viscosity values. This modification allows the Semantic Digital Twin to account for temperature variations when predicting fluid sloshing behaviour.

* **Question 3.2:** Simulate an environment where the ambient temperature rises to $28^\circ\text{C}$. How does this change the predicted wave height for a tilt rate of $0.15 \, \text{rad/s}$? Does the Warrant Gate trigger a safety violation?

ANS: I tested the effect of temperature on fluid sloshing while keeping the tilt rate constant at 0.15 rad/s. At 20°C, the predicted wave height was 6.26 mm, while at 28°C, it increased to 6.94 mm. This increase of 0.68 mm demonstrates that higher temperatures reduce effective fluid viscosity, allowing greater fluid movement and increasing the predicted wave height. Therefore, the Semantic Digital Twin accounts for temperature variations when predicting fluid behaviour.

### Task 4: Expanding the Warrant Gate Checklist
The current Warrant Gate evaluates six core criteria (*grounded, feasible, safe, monitorable, repairable, recordable*). 
* **Question 4.1:** Add a seventh criterion called **`energetically_viable`** to the Warrant Gate dictionary checks. It must evaluate whether the estimated battery drain for the requested trajectory stays below $15\%$ of total capacity.

ANS: I extended the Warrant Gate with a seventh criterion called `energetically_viable`. This criterion checks whether the estimated battery drain for the proposed trajectory is strictly below 15% of total battery capacity and does not exceed the robot's currently available battery level. The check is performed using the estimated battery drain in the `ParameterPacket` and the battery level stored in the `BeliefState`. This ensures that the robot only proceeds when sufficient energy is available for the requested action.

 **Question 4.2:** Introduce a low-battery state into the `BeliefState` and demonstrate how a failed `energetically_viable` check redirects the system workflow to request a recharge action rather than proceeding with the coffee delivery.

ANS: I introduced a low-battery state with 15% remaining battery capacity and tested a trajectory with an estimated battery drain of 16%. The Warrant Gate rejected the proposal because the estimated battery drain exceeded the permitted 15% threshold and the available battery level. The safety criterion also failed during validation. The workflow then redirected the robot to request a recharge instead of proceeding with coffee delivery. This demonstrates how energy viability and safety checks influence action selection.

### Task 5: Post-Hoc Epistemic Audit Analysis
A core pillar of trustworthy robotics is the ability to audit *why* a decision was made or rejected.
* **Question 5.1:** Write a post-processing function (`audit_analyzer(audit_trail)`) that parses the recorded episodic log. It should compute and print:
  1. The total number of rejected proposals versus approved executions.
  2. The most frequent warrant criterion responsible for rejection.
  3. The average variance between predicted forces and observed physical outcomes.

ANS: I implemented the record_trace() method inside the ExecutionEngine class to maintain an episodic audit trail.

Each audit entry records:
- User request: Original command given to the robot.
- Commitment decision: Whether the proposed action was approved or rejected.
- Action parameters: Proposed velocity and tilt rate.
- Warrant verdict: Predictions and results of all Warrant Gate checks.
- Execution outcome: Actual force, fluid wave, transferred volume and spill status, when execution occurs.
This allows the robot's decision-making process to be reconstructed and inspected, including rejected proposals.
I implemented `audit_analyzer(audit_trail)` to count approved and rejected proposals, identify the most frequent failed Warrant Gate criteria, and calculate the population variance of absolute force prediction errors when execution data is available. In the current test, both proposals were rejected, and no physical execution took place. Therefore, the force prediction error variance is reported as None.

* **Question 5.2:** Discuss in your lab report: *Why does maintaining separate records for `asked`, `belief`, `committed`, and `observed` solve the epistemological gap that pure data-driven imitation learning cannot address?*

ANS: I extended the audit trail to record four separate elements: asked, which stores the original user instruction and interpreted goal; belief, which stores the robot's internal representation of the environment; committed, which records the proposed parameters and Warrant Gate decision; and observed, which stores the actual sensor measurements. The two audit entries show how the system records rejected proposals, including their predictions and failed safety and energy checks. This separation makes the decision-making process traceable and helps identify where a failure occurred.

Conclusion: The audit trail provides traceability and accountability by recording the robot's proposals, decisions and outcomes. The analyzer helps identify recurring failure conditions and evaluate prediction accuracy when physical execution data is available.

## Part 3: Robot Anatomy (0.5 Pt)

**Title:** Links Vs Joints, Actuators Vs Sensors, Effectors Vs Bases Vs Torsos Vs Heads, Wheels Vs Legs Vs Grippers, Proprioception Vs Exteroception Vs Interoception

![Alternative Text](robot-body.jpg)

### Questions

Identify the key functional parts of a robot based on the above picture and briefly indicate how they support the execution of function in the above architecture.

ANS:  Part 3: Robot Anatomy

The diagram illustrates the main mechanical, sensing and actuation components of a mobile manipulator robot.

1. **Links and Joints:** Links are the rigid segments forming the robot's arms and body, while joints connect these segments and allow rotational or translational movement. They enable the robot to position its arm and manipulate objects.

2. **Actuators and Sensors:** Actuators, such as the motors shown in the diagram, generate movement in the robot's joints and wheels. Sensors collect information about the robot and its surroundings, supporting perception and decision-making.

3. **Effectors, Base, Torso and Head:**
   - **Effectors:** The robot's end-effectors, particularly its grippers, interact directly with objects.
   - **Base:** The bottom platform supports the robot and houses its mobility system.
   - **Torso:** The central body supports the arms and connects the upper body to the base.
   - **Head:** The upper section carries sensing equipment, including cameras, for observing the environment.

4. **Wheels, Legs and Grippers:**
   - **Wheels:** Enable the robot to navigate its environment.
   - **Legs:** Provide articulated locomotion where a leg-based movement system is used. The illustrated robot primarily uses a wheeled base.
   - **Grippers:** Allow the robot to grasp, hold and manipulate objects, such as the items shown in the kitchen scene.

5. **Proprioception, Exteroception and Interoception:**
   - **Proprioception:** Provides information about the robot's own configuration and movement, such as joint positions and wheel rotation.
   - **Exteroception:** Provides information about the external environment through sensors such as the camera and laser distance sensor shown in the diagram.
   - **Interoception:** Provides information about the robot's internal state, such as battery level, motor condition and temperature.

**Connection to the Integrated Intelligent Systems Architecture:**

These physical components support the architecture by providing the robot with the ability to sense and act. Sensors provide observations to the Sensing and Perception components. The Interpretation and Planning components determine the required actions, while the Semantic Digital Twin predicts their consequences. The Warrant Gate checks whether the proposed actions are safe before execution. Finally, the Acting component sends commands to the actuators, joints, wheels and grippers to perform the task. The resulting observations and execution outcomes can be recorded for monitoring and analysis.

Thus, the robot's anatomy provides the physical foundation for sensing, mobility and object manipulation, allowing the intelligent architecture to execute tasks in the real world.

---

# Submission guidelines

- Deadline: 03.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (Create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put all the answers in the notebook. Any other media related to your answer should be inserted in the notebook as well. The notebook and all related media should then be put into a folder called **Assignment1**. 